<a href="https://colab.research.google.com/github/feronika-lab/Risk-Based-Motor-Insurance-Premium-Pricing-Engine-freMTPL2-with-Tweedie-and-XGBoost/blob/main/04_premium_calculator_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

try:
    df = pd.read_csv('df_scored_glm.csv')
    print("Kolom-kolom dalam DataFrame:")
    print(df.columns.tolist())
except FileNotFoundError:
    print("Error: File 'df_scored_glm.csv' tidak ditemukan. Pastikan file sudah diunggah.")
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

Kolom-kolom dalam DataFrame:
['IDpol', 'ClaimNb', 'Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'Area', 'Density', 'Region', 'ClaimAmount', 'ClaimAmount_Capped', 'LogDensity', 'VehAge_Group', 'Area_Encoded', 'VehBrand_Encoded', 'VehGas_Encoded', 'Region_Encoded', 'VehAge_Group_Encoded', 'PurePremium_Capped', 'GLM_Prediction']


Blok kode ini akan melatih ulang model GLM secara cepat, lalu menyediakan sebuah fungsi di mana Anda tinggal memasukkan data nasabah baru, dan tada!—harga premi langsung muncul.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm            # Tambahkan ini untuk mengakses families
import statsmodels.formula.api as smf    # Digunakan untuk formula string

# 1. SETUP MODEL (Re-train menggunakan pemisahan modul yang benar)
df_calc = pd.read_csv('df_scored_glm.csv')

# Pastikan data bersih dari baris kosong (NaN) agar tidak error saat fit
df_calc = df_calc.dropna(subset=['PurePremium_Capped', 'Exposure'])

formula = "PurePremium_Capped ~ BonusMalus + DrivAge + VehPower + LogDensity + C(Area) + C(VehGas) + C(VehAge_Group)"

# Perbaikan: Gunakan sm.families (bukan smf.families)
tweedie_link = smf.glm(formula=formula, data=df_calc,
                       family=sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5),
                       var_weights=df_calc['Exposure']).fit()

def hitung_premi_otomatis(nama, usia, bonus_malus, power_mesin, kepadatan_kota, area, bensin, kelompok_usia_mobil):
    # Menghitung LogDensity
    log_dens = np.log(kepadatan_kota)

    # Membuat data nasabah baru
    nasabah_baru = pd.DataFrame({
        'DrivAge': [usia],
        'BonusMalus': [bonus_malus],
        'VehPower': [power_mesin],
        'LogDensity': [log_dens],
        'Area': [area],
        'VehGas': [bensin],
        'VehAge_Group': [kelompok_usia_mobil]
    })

    # Prediksi
    estimasi_premi = tweedie_link.predict(nasabah_baru)[0]

    print(f"--- 📝 HASIL ESTIMASI PREMI UNTUK: {nama} ---")
    print(f"Profil: Usia {usia} thn, BM {bonus_malus}, Area {area}")
    print(f"Estimasi Premi Murni: € {estimasi_premi:.2f} per tahun")
    print("-" * 45)

# ==========================================
# 🚀 TEST RUN
# ==========================================
hitung_premi_otomatis(
    nama="Budi",
    usia=20,
    bonus_malus=120,
    power_mesin=7,
    kepadatan_kota=15000,
    area='F',
    bensin='Regular',
    kelompok_usia_mobil='Old'
)
# Contoh 2: Pak Slamet (Senior, Aman/Bonus 50, Tinggal di Desa Sepi)
hitung_premi_otomatis(
    nama="Pak Slamet",
    usia=55,
    bonus_malus=50,
    power_mesin=4,
    kepadatan_kota=50,
    area='A',
    bensin='Diesel',
    kelompok_usia_mobil='New'
)

--- 📝 HASIL ESTIMASI PREMI UNTUK: Budi ---
Profil: Usia 20 thn, BM 120, Area F
Estimasi Premi Murni: € 333.18 per tahun
---------------------------------------------
--- 📝 HASIL ESTIMASI PREMI UNTUK: Pak Slamet ---
Profil: Usia 55 thn, BM 50, Area A
Estimasi Premi Murni: € 58.03 per tahun
---------------------------------------------
